Celda 1 (markdown): título. Indica que se ejecuta en orden sobre un servidor de Colab con CPU.

In [7]:
import os
import sys

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "03DataCollectionAndEnvironmentSetup"

if not os.path.exists("/content/repo"):
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo pull

%cd /content/repo
!pip install -q -r requirements.txt
sys.path.insert(0, "/content/repo/src")

Already up to date.
/content/repo
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [4]:
!kaggle datasets download -d kaggle/pokemon-tcg-ai-battle-episodes-2026-06-18 -p /content/raw --unzip
!ls /content/raw | head
!du -sh /content/raw

Dataset URL: https://www.kaggle.com/datasets/kaggle/pokemon-tcg-ai-battle-episodes-2026-06-18
License(s): CC0-1.0
100% 708M/708M [00:06<00:00, 123MB/s]  

80386449.json
80386463.json
80386478.json
80386479.json
80386483.json
80386486.json
80386488.json
80386490.json
80386493.json
80386500.json
21G	/content/raw


In [5]:
import glob, json, os

files = sorted(glob.glob("/content/raw/*.json"))
print("archivos:", len(files))

path = files[0]
print("tamaño:", round(os.path.getsize(path) / 1e6, 2), "MB")

with open(path) as f:
    data = json.load(f)

def resumen(x, nivel=0, max_nivel=3):
    pad = "  " * nivel
    if isinstance(x, dict):
        for k, v in list(x.items())[:15]:
            print(f"{pad}{k}: {type(v).__name__}" + (f" (len {len(v)})" if hasattr(v, "__len__") and not isinstance(v, str) else ""))
            if nivel < max_nivel:
                resumen(v, nivel + 1, max_nivel)
    elif isinstance(x, list) and x:
        print(f"{pad}[0]: {type(x[0]).__name__}")
        if nivel < max_nivel:
            resumen(x[0], nivel + 1, max_nivel)

resumen(data)

archivos: 6532
tamaño: 2.54 MB
configuration: dict (len 4)
  actTimeout: int
  episodeSteps: int
  runTimeout: int
  seed: int
description: str
id: str
info: dict (len 4)
  Agents: list (len 2)
    [0]: dict
      Name: str
      ThumbnailUrl: NoneType
  EpisodeId: int
  LiveVideoPath: NoneType
  TeamNames: list (len 2)
    [0]: str
module_version: str
name: str
rewards: list (len 2)
  [0]: int
schema_version: int
specification: dict (len 6)
  action: dict (len 3)
    default: list (len 0)
    description: str
    type: str
  agents: list (len 1)
    [0]: int
  configuration: dict (len 3)
    actTimeout: dict (len 4)
      default: int
      description: str
      minimum: int
      type: str
    episodeSteps: dict (len 4)
      default: int
      description: str
      minimum: int
      type: str
    runTimeout: dict (len 4)
      default: int
      description: str
      minimum: int
      type: str
  info: dict (len 0)
  observation: dict (len 2)
    remainingOverageTime: dict (len

In [6]:
import json

with open(files[0]) as f:
    data = json.load(f)

print("rewards:", data["rewards"])
print("statuses:", data["statuses"])
print("pasos:", len(data["steps"]))
print("agentes:", len(data["info"]["Agents"]), "| claves de Agents[0]:", list(data["info"]["Agents"][0].keys()))


def corto(v, n=150):
    s = json.dumps(v, ensure_ascii=False)
    return s if len(s) <= n else s[:n] + f"... [{len(s)} caracteres]"


def abrir(x, nombre, nivel=0, max_nivel=3):
    pad = "  " * nivel
    if isinstance(x, dict):
        for k, v in x.items():
            tipo = type(v).__name__
            largo = f" len={len(v)}" if isinstance(v, (list, dict, str)) else ""
            print(f"{pad}{nombre}.{k}: {tipo}{largo}  {corto(v) if not isinstance(v, (dict, list)) else ''}")
            if isinstance(v, (dict, list)) and nivel < max_nivel:
                abrir(v, k, nivel + 1, max_nivel)
    elif isinstance(x, list) and x:
        print(f"{pad}{nombre}[0]: {type(x[0]).__name__}")
        if nivel < max_nivel:
            abrir(x[0], f"{nombre}[0]", nivel + 1, max_nivel)


print("\n=== steps[0][0].observation ===")
abrir(data["steps"][0][0]["observation"], "obs")

print("\n=== steps[0][1].observation ===")
abrir(data["steps"][0][1]["observation"], "obs")

mitad = len(data["steps"]) // 2
print(f"\n=== steps[{mitad}][0] (mitad de la partida) ===")
print("action:", corto(data["steps"][mitad][0]["action"]))
print("status:", data["steps"][mitad][0]["status"], "| reward:", data["steps"][mitad][0]["reward"])
abrir(data["steps"][mitad][0]["observation"], "obs")

print("\n=== visualize[0] ===")
abrir(data["steps"][0][0]["visualize"][0], "vis", max_nivel=2)

print("\n=== último paso ===")
for j in range(2):
    ult = data["steps"][-1][j]
    print(j, "status:", ult["status"], "| reward:", ult["reward"])

rewards: [-1, 1]
statuses: ['DONE', 'DONE']
pasos: 100
agentes: 2 | claves de Agents[0]: ['Name', 'ThumbnailUrl']

=== steps[0][0].observation ===
obs.current: NoneType  null
obs.logs: list len=0  
obs.remainingOverageTime: int  60
obs.search_begin_input: NoneType  null
obs.select: NoneType  null
obs.step: int  0

=== steps[0][1].observation ===
obs.current: NoneType  null
obs.logs: list len=0  
obs.remainingOverageTime: int  60
obs.search_begin_input: NoneType  null
obs.select: NoneType  null

=== steps[50][0] (mitad de la partida) ===
action: [2]
status: ACTIVE | reward: 0
obs.current: dict len=12  
  current.energyAttached: bool  true
  current.firstPlayer: int  0
  current.looking: NoneType  null
  current.players: list len=2  
    players[0]: dict
      players[0].active: list len=1  
      players[0].asleep: bool  false
      players[0].bench: list len=4  
      players[0].benchMax: int  5
      players[0].burned: bool  false
      players[0].confused: bool  false
      players[0